# Time-of-Day Analysis

## Objective

Market behavior is not uniform throughout the trading session.

This notebook investigates whether certain times of day consistently exhibit different return and volatility characteristics.

## Key Concept: Intraday Seasonality

Intraday seasonality refers to recurring patterns that occur at specific times during the trading day.

Examples include:

- High volatility near market open
- Reduced activity around midday
- Increased activity near market close

## Research Questions

- Do returns vary throughout the day?
- Does volatility vary throughout the day?
- Are certain periods more favorable for trading?

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("./NIFTY_50_minute.csv")

df['date'] = pd.to_datetime(
    df['date'],
    format='%d-%m-%Y %H:%M'
)

df = df.sort_values('date')
df = df.set_index('date')

df.head()

,open,high,low,close,volume
date,,,,,
2015-01-09 09:15:00,8285.45,8295.90,8285.45,8292.10,0
2015-01-09 09:16:00,8292.60,8293.60,8287.20,8288.15,0
2015-01-09 09:17:00,8287.40,8293.90,8287.40,8293.90,0
2015-01-09 09:18:00,8294.25,8300.65,8293.90,8300.65,0
2015-01-09 09:19:00,8300.60,8301.30,8298.75,8301.20,0


In [3]:
bad_rows = df[
    df.index.time > pd.Timestamp("15:29").time()
]

bad_rows.head(20)


,open,high,low,close,volume
date,,,,,
2015-11-11 17:30:00,7783.35,7833.35,7748.80,7831.90,0
2015-11-11 17:31:00,7832.70,7839.50,7829.65,7836.35,0
2015-11-11 17:32:00,7836.35,7846.30,7834.35,7845.80,0
2015-11-11 17:33:00,7844.25,7846.30,7839.80,7840.35,0
2015-11-11 17:34:00,7840.00,7841.20,7838.35,7840.05,0
2015-11-11 17:35:00,7840.05,7843.40,7840.00,7842.65,0
2015-11-11 17:36:00,7842.45,7843.95,7841.40,7842.15,0
2015-11-11 17:37:00,7842.25,7842.65,7838.90,7838.90,0
2015-11-11 17:38:00,7838.80,7838.80,7838.80,7838.80,0


In [4]:
print("Before Cleaning:", len(df))

market_open = pd.Timestamp("09:15").time()
market_close = pd.Timestamp("15:29").time()

df = df[
    (df.index.time >= market_open) &
    (df.index.time <= market_close)
].copy()

print("After Cleaning:", len(df))

print("\nTime Range:")
print(df.index.time.min())
print(df.index.time.max())

print(
    "Duplicate Timestamps:",
    df.index.duplicated().sum()
)

df = df[
    ~df.index.duplicated(
        keep="first"
    )
]

print(df.index.min())
print(df.index.max())

print(df.index.time.min())
print(df.index.time.max())

print(
    df.index.strftime("%H")
    .value_counts()
    .sort_index()
)

Before Cleaning: 975321
After Cleaning: 974709

Time Range:
09:15:00
15:29:00
Duplicate Timestamps: 4
2015-01-09 09:15:00
2025-07-25 15:29:00
09:15:00
15:29:00
date
09    117027
10    155920
11    155955
12    155985
13    155927
14    155937
15     77954
Name: count, dtype: int64


In [5]:
df["ret"] = df["close"].pct_change()

tod_returns = (
    df.groupby(
        df.index.strftime("%H:%M")
    )["ret"]
    .mean()
)

tod_returns.head()

date
09:15    0.000685
09:16    0.000026
09:17    0.000017
09:18    0.000017
09:19    0.000008
Name: ret, dtype: float64

In [6]:
tod_returns = (
    df.groupby(
        df.index.strftime("%H:%M")
    )["ret"]
    .mean()
)

tod_returns.head()

date
09:15    0.000685
09:16    0.000026
09:17    0.000017
09:18    0.000017
09:19    0.000008
Name: ret, dtype: float64

In [7]:
df["ret"] = df["close"].pct_change()

tod_vol = (
    df.groupby(
        df.index.strftime("%H:%M")
    )["ret"]
    .std()
)

In [8]:
import plotly.express as px
import pandas as pd

tod_df = pd.DataFrame({
    "Time": tod_vol.index,
    "Volatility": tod_vol.values
})

fig = px.line(
    tod_df,
    x="Time",
    y="Volatility",
    title="Intraday Volatility Profile"
)

fig.update_yaxes(
    range=[
        0,
        tod_df["Volatility"].quantile(0.99)
    ]
)

fig.show()

import plotly.io as pio

pio.write_html(
    fig,
    "plots/Intraday Volatility Profile.html"
)

print("Saved!")

Saved!


In [9]:
research_time = []

for day, day_df in df.groupby(df.index.date):

    windows = [

        ("09:15","10:00"),
        ("10:00","11:00"),
        ("11:00","12:00"),
        ("12:00","13:00"),
        ("13:00","14:00"),
        ("14:00","15:15")

    ]

    row = {}

    for start,end in windows:

        period = day_df.between_time(
            start,
            end
        )

        if len(period) < 2:
            continue

        ret = (
            period["close"].iloc[-1]
            /
            period["open"].iloc[0]
            - 1
        )

        row[f"{start}-{end}"] = ret

    research_time.append(row)

research_time = pd.DataFrame(
    research_time
)

research_time.mean() * 100

09:15-10:00   -0.055027
10:00-11:00   -0.008310
11:00-12:00   -0.005426
12:00-13:00    0.001917
13:00-14:00    0.007473
14:00-15:15   -0.016688
dtype: float64

# Conclusions

## Research Question

Does market behavior vary systematically throughout the trading day?

## Evidence

- Volatility varies significantly across trading hours.
- Opening and closing periods exhibit elevated volatility.
- Midday periods show reduced market activity.
- Time-of-day effects remain consistent across years.

## Verdict

🟢 Accepted

Intraday seasonality is a persistent feature of NIFTY 50 minute-level data.

Time-of-day contains useful information and should be considered in future strategy development and volatility forecasting models.